# EX: Bayes' Rule

In autonomous tactical systems, sensor alerts rarely indicate ground truth with 100% certainty. An AI agent begins with a prior belief about an operational state (such as a hardware failure, an incoming missile, or cyber infiltration). When an onboard detector sounds an alert, the system must apply **Bayes' Rule** to compute the **Posterior Probability**—the updated probability that the tactical event is actually occurring given the observed alert.

This lab demonstrates how to programmatically implement Bayes' Rule, perform belief updates, and evaluate how the underlying **Base Rate** of an event drastically influences the reliability of an alert.

An autonomous Space Force satellite monitors its internal propulsion valves.

* **Prior Probability of Valve Failure ($P(F)$):** $0.10$ (10% of satellites experience this failure in a standard orbit).


* **Detector Likelihood ($P(A \mid F)$):** $0.80$ (The diagnostic system detects 80% of true failures).


* **Marginal Probability of Alert ($P(A)$):** $0.20$ (The detector sounds an alarm 20% of the time across all operational states).


## Lab Steps

The Python script executes the following computational steps:

1. **Define System Parameters:** Assign the prior probability $P(F)$, likelihood $P(A \mid F)$, and total evidence probability $P(A)$.

2. **Implement Bayes' Rule:** Define an inference function to compute:

$$P(F \mid A) = \frac{P(A \mid F) \cdot P(F)}{P(A)}$$

3. **Compute the Posterior Probability:** Calculate the updated probability of failure once an alarm sounds.

4. **Base Rate Sensitivity Analysis:** Simulate how the posterior probability shifts when the baseline failure rate varies from rare ($1\%$) to frequent ($30\%$), holding sensor sensitivity constant.

In [1]:
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Step 1: Define Baseline Parameters
# -------------------------------------------------------------------------
prior_failure = 0.10             # P(F): Baseline probability of valve failure
likelihood_alert = 0.80          # P(A|F): Probability detector alerts given failure
marginal_alert = 0.20            # P(A): Overall probability of an alert sounding

# -------------------------------------------------------------------------
# Step 2: Implement Bayes' Rule Function
# -------------------------------------------------------------------------
def compute_posterior(prior, likelihood, marginal_evidence):
    """
    Computes P(Hypothesis | Evidence) using Bayes' Rule.
    """
    return (likelihood * prior) / marginal_evidence

# -------------------------------------------------------------------------
# Step 3: Compute Single-Event Posterior Probability
# -------------------------------------------------------------------------
posterior_failure = compute_posterior(prior_failure, likelihood_alert, marginal_alert)

print("=" * 55)
print("TACTICAL BELIEF UPDATING ENGINE")
print("=" * 55)
print(f"Prior Probability of Failure P(F):           {prior_failure:.2f}")
print(f"Detector Likelihood P(A|F):                   {likelihood_alert:.2f}")
print(f"Marginal Alert Rate P(A):                     {marginal_alert:.2f}")
print("-" * 55)
print(f"Updated Posterior Probability P(F|A):         {posterior_failure:.2f} ({posterior_failure * 100:.1f}%)")
print("=" * 55 + "\n")

# -------------------------------------------------------------------------
# Step 4: Base Rate Sensitivity Analysis
# -------------------------------------------------------------------------
# Assume false positive rate on healthy components is P(A | ~F) = 0.1333
# to model P(A) = P(A|F)P(F) + P(A|~F)P(~F) across different base rates.
false_positive_rate = 0.1333
base_rates = [0.01, 0.05, 0.10, 0.20, 0.30]

sensitivity_results = []

for base_p in base_rates:
    # Compute marginal P(A) dynamically for each base rate
    p_not_f = 1.0 - base_p
    p_alert = (likelihood_alert * base_p) + (false_positive_rate * p_not_f)
    
    # Calculate updated posterior
    p_f_given_a = compute_posterior(base_p, likelihood_alert, p_alert)
    
    sensitivity_results.append({
        "Base Rate P(F)": f"{base_p * 100:.1f}%",
        "Marginal P(A)": f"{p_alert:.4f}",
        "Posterior P(F|A)": f"{p_f_given_a * 100:.2f}%"
    })

df_sensitivity = pd.DataFrame(sensitivity_results)
print("=== BASE RATE SENSITIVITY TABLE ===")
print(df_sensitivity.to_string(index=False))

TACTICAL BELIEF UPDATING ENGINE
Prior Probability of Failure P(F):           0.10
Detector Likelihood P(A|F):                   0.80
Marginal Alert Rate P(A):                     0.20
-------------------------------------------------------
Updated Posterior Probability P(F|A):         0.40 (40.0%)

=== BASE RATE SENSITIVITY TABLE ===
Base Rate P(F) Marginal P(A) Posterior P(F|A)
          1.0%        0.1400            5.72%
          5.0%        0.1666           24.00%
         10.0%        0.2000           40.01%
         20.0%        0.2666           60.01%
         30.0%        0.3333           72.01%


# Interpreting the Results

**Likelihood vs. Posterior Discrepancy:** The diagnostic sensor boasts an $80\%$ detection accuracy ($P(A \mid F) = 0.80$). However, when the alarm triggers in flight, there is only a $40\%$ chance that a failure is actually taking place ($P(F \mid A) = 0.40$). An operator who confuses the sensor's sensitivity with the true posterior probability will overestimate the urgency by a factor of two.  

**The Base Rate Fallacy:** The sensitivity table reveals that when a failure is rare ($1\%$ base rate), an alert yields only a $5.71\%$ posterior probability of true failure. Because $99\%$ of the operational population is healthy, even a small false-alarm percentage on healthy components produces a large volume of false alerts that overshadow true detections.  

**Operational Decision Thresholds:** Automated defense systems cannot trigger critical fail-safe shutdowns based solely on a high detection rate ($P(A \mid F)$). The AI must condition on the prior base rate to avoid grounding mission assets on false alarms. 